In [3]:
from langchain_community.document_loaders import ArxivLoader

In [4]:
docs = ArxivLoader(
    query="1605.08386",load_max_docs = 2).load()

## RecursiveCharacterTextSplitter

Imports the RecursiveCharacterTextSplitter, a utility that splits long text into smaller chunks using a prioritized set of separators (e.g., "\n\n", "\n", " ", "") to preserve semantic boundaries.

### When to use
- Prepare long documents for embedding or retrieval
- Ensure chunks fit model/context limits while keeping coherence

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


### Key params
- `chunk_size` — max tokens/characters per chunk (e.g., 500)  
- `chunk_overlap` — overlap between consecutive chunks (e.g., 50)  
- `separators` — optional list of separators to try in order

### Example
```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# For plain text
chunks = text_splitter.split_text(long_text)

# For LangChain Document objects
chunks = text_splitter.split_documents(documents)
```

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_documents = text_splitter.split_documents(docs)

In [7]:
print(final_documents [0])

page_content='arXiv:1605.08386v1  [math.CO]  26 May 2016
HEAT-BATH RANDOM WALKS WITH MARKOV BASES
CAPRICE STANLEY AND TOBIAS WINDISCH
Abstract. Graphs on lattice points are studied whose edges come from a ﬁnite set of
allowed moves of arbitrary length. We show that the diameter of these graphs on ﬁbers of a
ﬁxed integer matrix can be bounded from above by a constant. We then study the mixing
behaviour of heat-bath random walks on these graphs. We also state explicit conditions' metadata={'Published': '2016-05-26', 'Title': 'Heat-bath random walks with Markov bases', 'Authors': 'Caprice Stanley, Tobias Windisch', 'Summary': 'Graphs on lattice points are studied whose edges come from a finite set of allowed moves of arbitrary length. We show that the diameter of these graphs on fibers of a fixed integer matrix can be bounded from above by a constant. We then study the mixing behaviour of heat-bath random walks on these graphs. We also state explicit conditions on the set of moves so that

### create_documents vs split_documents
- `create_documents`: use when you have raw strings (list[str]) — creates Document objects (and splits them if needed).  
- `split_documents`: use when you already have `Document` objects — splits those into smaller `Document` objects.  
- `split_text`: returns raw text chunks (list[str]) from a single string.

In [8]:
from langchain_community.document_loaders import TextLoader

In [9]:
loader = TextLoader('knowledge_copy.txt')

In [10]:
loader.load()

[Document(metadata={'source': 'knowledge_copy.txt'}, page_content='Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.\n\nLarge Language Models (LLMs) LLMs like GPT-4, Claude, and Gemini represent a massive leap in Natural Language Processing. These models are trained on vast datasets, allowing them to predict the next token in a sequence with remarkable accuracy. However, they face a common challenge: "hallucination," where the model generates confident but incorrect information.\n\nThe Role of LangChain LangChain is a framework designed to simplify the creation of applications using LLMs. It provides a standard interface for:\n\nChains: Linking different components together.\n\nMemory: Allowing models to remember past in

In [37]:
content1 = " "
with open('knowledge_copy.txt', 'r') as file:
    content1 = file.read()

In [38]:
content1

'Knowledge Base: The Evolution of AI\nArtificial Intelligence (AI) has transitioned from a theoretical concept to a transformative technology. At its core, AI aims to create systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, and decision-making.\n\nLarge Language Models (LLMs) LLMs like GPT-4, Claude, and Gemini represent a massive leap in Natural Language Processing. These models are trained on vast datasets, allowing them to predict the next token in a sequence with remarkable accuracy. However, they face a common challenge: "hallucination," where the model generates confident but incorrect information.\n\nThe Role of LangChain LangChain is a framework designed to simplify the creation of applications using LLMs. It provides a standard interface for:\n\nChains: Linking different components together.\n\nMemory: Allowing models to remember past interactions.\n\nRAG (Retrieval-Augmented Generation): Connecting LL

In [25]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
text = text_splitter.create_documents([content])

In [26]:
print(text)

[Document(metadata={}, page_content='Knowledge Base: The Evolution of AI'), Document(metadata={}, page_content='Artificial Intelligence (AI) has transitioned from a theoretical concept to a transformative'), Document(metadata={}, page_content='to a transformative technology. At its core, AI aims to create systems capable of performing tasks'), Document(metadata={}, page_content='of performing tasks that typically require human intelligence, such as visual perception, speech'), Document(metadata={}, page_content='perception, speech recognition, and decision-making.'), Document(metadata={}, page_content='Large Language Models (LLMs) LLMs like GPT-4, Claude, and Gemini represent a massive leap in'), Document(metadata={}, page_content='a massive leap in Natural Language Processing. These models are trained on vast datasets, allowing'), Document(metadata={}, page_content='datasets, allowing them to predict the next token in a sequence with remarkable accuracy. However,'), Document(metadata=

## Chunk overlap — short explanation

- **Definition:** Number of characters (or tokens for token-based splitters) repeated between consecutive chunks so context spans boundaries.  
- **Purpose:** Preserves continuity and prevents losing info at chunk edges — improves embeddings, retrieval, and LLM prompts.  
- **Units:** RecursiveCharacterTextSplitter uses **characters**; token-based splitters use **tokens**.  
- **Example:** `chunk_size=100`, `chunk_overlap=20`  
  - **Chunk 1:** chars 0–99  
  - **Chunk 2:** chars 80–179 (**chars 80–99 repeated**)  
- **Trade-off:** Larger overlap → **better context** but more **redundancy, storage, and compute**. Smaller overlap → less redundancy but possible **context loss**.  
- **Recommendation:** Start with overlap ≈ **10–20%** of `chunk_size` (e.g., **50** overlap for `chunk_size=500`) and adjust based on results.